# Capstone Project: End-to-End LLM Fine-Tuning & Deployment Pipeline

## Overview

This notebook ties together all four stages of the LLM Learning Roadmap into a single, production-grade pipeline. You will:

1. **Build** a domain-specific instruction dataset
2. **Fine-tune** a base model with QLoRA (4-bit quantized LoRA)
3. **Align** the model with human preferences using DPO
4. **Quantize** the final model with AWQ for efficient inference
5. **Deploy** the model behind a vLLM inference server
6. **Wrap** everything in a FastAPI gateway with auth and rate limiting
7. **Containerize** with Docker Compose
8. **Evaluate** performance against the base model
9. **Document** the deployment for production handoff

### Target Hardware

- GPU: NVIDIA RTX 3090 / 4090 (24 GB VRAM)
- RAM: 64 GB system memory recommended
- Storage: 100+ GB free (models are large)

### Project Directory Structure

```
capstone-project/
+-- data/
|   +-- raw/                  # Raw scraped documents
|   +-- processed/            # Cleaned & tokenized data
|   +-- instructions.jsonl    # Generated instruction dataset
|   +-- preferences.jsonl     # DPO preference pairs
+-- models/
|   +-- base/                 # Base model (e.g., Llama-3-8B)
|   +-- qlora-adapter/        # QLoRA adapter weights
|   +-- dpo-checkpoint/       # DPO fine-tuned checkpoint
|   +-- awq-quantized/        # AWQ 4-bit quantized model
+-- src/
|   +-- dataset.py            # Dataset generation & loading
|   +-- train_qlora.py        # QLoRA training script
|   +-- train_dpo.py          # DPO training script
|   +-- quantize_awq.py       # AWQ quantization script
|   +-- evaluate.py           # Evaluation script
|   +-- utils.py              # Shared utilities
+-- gateway/
|   +-- main.py               # FastAPI gateway
|   +-- auth.py               # API key authentication
|   +-- rate_limit.py         # Rate limiting config
|   +-- requirements.txt      # Gateway dependencies
+-- docker/
|   +-- Dockerfile.gateway    # Gateway container
|   +-- Dockerfile.vllm       # vLLM container
|   +-- docker-compose.yml    # Multi-service orchestration
+-- docs/
|   +-- DEPLOYMENT.md
|   +-- API.md
|   +-- TROUBLESHOOTING.md
+-- configs/
|   +-- qlora_config.yaml
|   +-- vllm_config.yaml
+-- requirements.txt
+-- README.md
```

> Before you start: Replace every `<YOUR_...>` placeholder with your actual values (paths, model names, API keys, etc.).

In [ ]:
# ============================================================
# Dependencies Installation
# ============================================================
!pip install --upgrade pip wheel setuptools

!pip install torch==2.4.1 transformers==4.44.2 datasets==2.21.0
!pip install accelerate==0.34.2 peft==0.12.0 bitsandbytes==0.43.3
!pip install trl==0.10.1 tokenizers==0.19.1
!pip install autoawq==0.2.6 vllm==0.6.1
!pip install fastapi==0.115.0 uvicorn==0.30.6 slowapi==0.1.9
!pip install python-dotenv==1.0.1 httpx==0.27.2
!pip install nltk==3.9.1 rouge-score==0.1.2 evaluate==0.4.3
!pip install wandb==0.17.7 tensorboard==2.17.1
!pip install pyyaml==6.0.2 tqdm==4.66.5 matplotlib==3.9.2 seaborn==0.13.2 pandas==2.2.2

print('[OK] All dependencies installed.')

In [ ]:
# ============================================================
# Imports & Global Config
# ============================================================
import os, json, yaml, random
from pathlib import Path
from typing import List, Dict, Optional

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import (
    AutoTokenizer, AutoModelForCausalLM, TrainingArguments,
    BitsAndBytesConfig, EarlyStoppingCallback,
)
from peft import (
    LoraConfig, get_peft_model, prepare_model_for_kbit_training,
    TaskType, PeftModel,
)
from datasets import Dataset
from trl import SFTTrainer, DPOTrainer, DPOConfig

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Paths -- CHANGE THESE
PROJECT_ROOT = Path("<YOUR_PROJECT_ROOT>")
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"

for d in [DATA_DIR, MODEL_DIR,
          DATA_DIR / "raw", DATA_DIR / "processed",
          MODEL_DIR / "base", MODEL_DIR / "qlora-adapter",
          MODEL_DIR / "dpo-checkpoint", MODEL_DIR / "awq-quantized"]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## Part 1: Domain-Specific Instruction Dataset

Build a synthetic instruction dataset. This template uses a **medical Q&A** domain -- swap the templates and seed data for your own domain.

**Format** (Alpaca/ShareGPT):
```json
{ "instruction": "...", "input": "...", "output": "..." }
```

In [ ]:
# ============================================================
# Part 1a: Instruction Templates & Seed Data
# ============================================================

# Domain-specific templates -- edit for your domain
INSTRUCTION_TEMPLATES = [
    "Explain {concept} in simple terms for a patient with no medical background.",
    "What are the potential side effects of {medication}?",
    "A patient presents with {symptoms}. What is the differential diagnosis?",
    "Describe the standard treatment protocol for {condition}.",
    "Compare {drug_a} and {drug_b} for treating {condition}.",
    "What lab tests should be ordered when a patient has {symptoms}?",
    "Summarize the latest clinical guidelines for managing {condition}.",
    "What are the contraindications for {medication}?",
    "Explain the pathophysiology of {condition} step by step.",
    "A patient asks: '{question}'. Draft a compassionate, accurate response.",
]

# Seed data pools -- expand for your domain
SEED_DATA = {
    "concept": [
        "type 2 diabetes", "hypertension", "atherosclerosis",
        "rheumatoid arthritis", "COPD", "hypothyroidism",
        "atrial fibrillation", "chronic kidney disease", "osteoarthritis", "migraine"
    ],
    "medication": [
        "metformin", "lisinopril", "atorvastatin", "levothyroxine",
        "metoprolol", "omeprazole", "amlodipine", "gabapentin",
        "sertraline", "albuterol", "warfarin", "insulin glargine"
    ],
    "symptoms": [
        "persistent dry cough and fatigue",
        "sudden chest pain with shortness of breath",
        "progressive joint pain and morning stiffness",
        "chronic headache with visual aura",
        "unexplained weight loss and polyuria",
        "intermittent abdominal pain with bloating",
        "persistent low back pain radiating to the leg",
        "recurrent fever of unknown origin"
    ],
    "condition": [
        "type 2 diabetes mellitus", "essential hypertension",
        "major depressive disorder", "gastroesophageal reflux disease",
        "chronic obstructive pulmonary disease", "atrial fibrillation",
        "iron deficiency anemia", "osteoarthritis of the knee"
    ],
    "drug_a": ["metformin", "lisinopril", "atorvastatin", "sertraline"],
    "drug_b": ["glipizide", "losartan", "rosuvastatin", "escitalopram"],
    "question": [
        "Is it safe to take ibuprofen with my blood pressure medication?",
        "How long will it take for my antidepressant to start working?",
        "Should I take my metformin before or after meals?",
        "Can I drink alcohol while on warfarin?",
        "What vaccines do I need if I have diabetes?"
    ],
}

print(f"Loaded {len(INSTRUCTION_TEMPLATES)} templates with "
      f"{sum(len(v) for v in SEED_DATA.values())} seed values across "
      f"{len(SEED_DATA)} categories")

In [ ]:
# ============================================================
# Part 1b: Generate 50 Examples
# ============================================================

def generate_instruction(idx: int) -> Dict[str, str]:
    """Generate a single instruction example by filling a template."""
    template = INSTRUCTION_TEMPLATES[idx % len(INSTRUCTION_TEMPLATES)]
    instruction = template
    for key, values in SEED_DATA.items():
        placeholder = "{" + key + "}"
        if placeholder in instruction:
            instruction = instruction.replace(placeholder, random.choice(values), 1)
    return {
        "instruction": instruction,
        "input": "",
        "output": f"[Reference answer for: {instruction[:80]}...]"  # fill with real outputs
    }

def generate_dataset(n_examples: int = 50, seed: int = 42) -> List[Dict]:
    """Generate n_examples instruction-tuning examples."""
    random.seed(seed)
    return [generate_instruction(i) for i in range(n_examples)]

# Generate
raw_dataset = generate_dataset(n_examples=50)
hf_dataset = Dataset.from_list(raw_dataset)
print(f"Generated {len(hf_dataset)} examples")
print(f"Columns: {hf_dataset.column_names}")
print(f"\nSample:\n{json.dumps(raw_dataset[0], indent=2)}")

In [ ]:
# ============================================================
# Part 1c: Data Validation & Save
# ============================================================

def validate_dataset(dataset: Dataset) -> Dict:
    """Run integrity checks on the instruction dataset."""
    report = {"total_examples": len(dataset), "errors": [], "warnings": []}
    for i, example in enumerate(dataset):
        inst = example.get("instruction", "")
        out  = example.get("output", "")
        if not inst.strip():
            report["errors"].append(f"Example {i}: empty instruction")
        if not out.strip():
            report["errors"].append(f"Example {i}: empty output")
        if len(inst.split()) < 3:
            report["warnings"].append(f"Example {i}: short instruction ({len(inst)} chars)")
        if "{" in inst or "{" in out:
            report["errors"].append(f"Example {i}: unfilled placeholder")
    report["valid"] = len(report["errors"]) == 0
    return report

validation = validate_dataset(hf_dataset)
print(json.dumps(validation, indent=2))

# Save
hf_dataset.to_json(str(DATA_DIR / "instructions.jsonl"))
print(f"\n[OK] Dataset saved to {DATA_DIR / 'instructions.jsonl'}")

# Train/val split
split = hf_dataset.train_test_split(test_size=0.2, seed=SEED)
split["train"].to_json(str(DATA_DIR / "train.jsonl"))
split["val"].to_json(str(DATA_DIR / "val.jsonl"))
print(f"[OK] Train: {len(split['train'])}, Val: {len(split['val'])}")

---
## Part 2: QLoRA Fine-Tuning

QLoRA combines **4-bit NF4 quantization** with LoRA adapters to fine-tune large models on consumer GPUs.

- 4-bit NF4: ~4 bits per weight parameter
- LoRA: small trainable adapters (< 1% of total parameters)
- Double quantization: further compresses quantization constants
- VRAM estimate: ~8-12 GB for a 7-8B model on a 24 GB GPU

Replace `<YOUR_BASE_MODEL>` with your actual base model ID.

In [ ]:
# ============================================================
# Part 2a: QLoRA Config
# ============================================================

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Training arguments
training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "qlora-adapter"),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.001,
    max_grad_norm=0.3,
    logging_steps=10,
    logging_dir=str(PROJECT_ROOT / "logs"),
    report_to="wandb",  # or "tensorboard"
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    tf32=True,
    seed=SEED,
    remove_unused_columns=False,
)

print("[OK] QLoRA configs ready.")
print(f"  LoRA rank: r={lora_config.r}, alpha={lora_config.lora_alpha}")
print(f"  Batch size: {training_args.per_device_train_batch_size} x {training_args.gradient_accumulation_steps} = effective {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")

In [ ]:
# ============================================================
# Part 2b: Model Loading & Training (skeleton)
# ============================================================
# Uncomment to run -- requires 8-12 GB VRAM

BASE_MODEL_ID = "<YOUR_BASE_MODEL>"  # e.g., meta-llama/Meta-Llama-3-8B

# Load tokenizer
# tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "right"

# Load 4-bit model
# model = AutoModelForCausalLM.from_pretrained(
#     BASE_MODEL_ID,
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True,
#     torch_dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
# )

# Prepare for k-bit training + apply LoRA
# model = prepare_model_for_kbit_training(model)
# model.gradient_checkpointing_enable()
# model = get_peft_model(model, lora_config)
# model.print_trainable_parameters()

# Train
# trainer = SFTTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=split["train"],
#     eval_dataset=split["val"],
#     tokenizer=tokenizer,
#     max_seq_length=2048,
# )
# trainer.train()
# trainer.save_model()

print("[INFO] Training skeleton ready. Uncomment code blocks to train.")

In [ ]:
# ============================================================
# Part 2c: Training Loss Visualization
# ============================================================

def plot_loss_curves(log_history: List[Dict] = None):
    """Plot training and validation loss from trainer logs."""
    if log_history is None:
        # Placeholder data for demonstration
        print("[INFO] Using placeholder data. Replace with real training logs.")
        steps = np.arange(0, 500, 10)
        np.random.seed(42)
        train_loss = 2.5 * np.exp(-steps / 200) + 0.3 + np.random.normal(0, 0.05, len(steps))
        val_loss = 2.5 * np.exp(-steps / 200) + 0.4 + np.random.normal(0, 0.08, len(steps))
    else:
        train_steps = [e["step"] for e in log_history if "loss" in e]
        train_loss  = [e["loss"] for e in log_history if "loss" in e]
        val_steps    = [e["step"] for e in log_history if "eval_loss" in e]
        val_loss     = [e["eval_loss"] for e in log_history if "eval_loss" in e]
        steps = val_steps  # align to eval frequency

    fig, ax = plt.subplots(figsize=(10, 5))
    if log_history is None:
        ax.plot(steps, train_loss, label="Train Loss", color="#2196F3", linewidth=1.5)
        ax.plot(steps, val_loss, label="Val Loss", color="#FF5722", linewidth=1.5)
    else:
        ax.plot(train_steps, train_loss, label="Train Loss", marker=".", ms=3, color="#2196F3")
        ax.plot(val_steps, val_loss, label="Val Loss", marker=".", ms=3, color="#FF5722")
    ax.set_xlabel("Training Steps"); ax.set_ylabel("Loss")
    ax.set_title("QLoRA Fine-Tuning: Training & Validation Loss")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / "training_loss.png", dpi=150)
    plt.show()

plot_loss_curves()

---
## Part 3: DPO (Direct Preference Optimization) Alignment

DPO directly optimizes a policy to match human preferences without a separate reward model.

**Data format**: Each row has a `prompt`, `chosen` (better response), and `rejected` (worse response).

In [ ]:
# ============================================================
# Part 3a: Preference Data Creation
# ============================================================

def create_preference_pair(
    instruction: str, input_text: str,
    good_response: str, bad_response: str
) -> Dict[str, str]:
    """Create a single DPO preference pair."""
    user_content = instruction
    if input_text:
        user_content += f"\n\n{input_text}"
    prompt = (
        f"<|im_start|>system\n"
        f"You are a helpful assistant.\n"
        f"<|im_end|>\n"
        f"<|im_start|>user\n"
        f"{user_content}\n"
        f"<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    return {"prompt": prompt, "chosen": good_response, "rejected": bad_response}

# Synthetic preference pairs -- replace with real human/LLM-judge annotations
PREFERENCE_TEMPLATES = [
    {
        "instruction": "Explain type 2 diabetes to a newly diagnosed patient.",
        "chosen": (
            "Type 2 diabetes is a condition where your body has trouble using insulin effectively. "
            "Insulin is a hormone that helps sugar move from your blood into your cells for energy. "
            "In type 2 diabetes, your cells become resistant to insulin, causing sugar buildup. "
            "This can be managed with lifestyle changes, oral medications like metformin, and "
            "sometimes insulin therapy. With proper management, many people live full, healthy lives."
        ),
        "rejected": "Diabetes is when you have too much sugar. Just stop eating sugar. No need to worry.",
    },
    {
        "instruction": "A patient asks if they can stop taking their blood pressure medication.",
        "chosen": (
            "I understand the desire to stop medication, but please discuss this with your doctor first. "
            "Suddenly stopping blood pressure medication can cause a dangerous rebound spike that "
            "increases your risk of stroke and heart attack. If you are experiencing side effects, "
            "your doctor can help create a safe tapering plan. Never stop antihypertensives abruptly."
        ),
        "rejected": "Sure, if you feel fine you can stop taking them. These drugs are overprescribed anyway.",
    },
    {
        "instruction": "What are the warning signs of a stroke?",
        "chosen": (
            "Use the FAST acronym: Face drooping, Arm weakness, Speech difficulty, Time to call "
            "emergency services. Other signs include sudden confusion, trouble seeing in one or both "
            "eyes, sudden severe headache, and trouble walking. Call emergency services immediately "
            "if you notice any of these -- every minute matters for treatment outcomes."
        ),
        "rejected": "If someone is having a stroke, just give them aspirin and wait to see if they improve.",
    },
    {
        "instruction": "What lifestyle changes help manage type 2 diabetes?",
        "chosen": (
            "Key lifestyle changes include: balanced diet with controlled carbs, 150+ min/week of "
            "moderate exercise, maintaining healthy weight, regular blood glucose monitoring, "
            "quitting smoking, limiting alcohol, stress management, and 7-9 hours of sleep nightly. "
            "These work alongside medication to improve insulin sensitivity."
        ),
        "rejected": "Just eat less and exercise more. That is really all there is to it.",
    },
    {
        "instruction": "Describe how metformin works.",
        "chosen": (
            "Metformin primarily reduces hepatic glucose production by activating AMPK, which "
            "inhibits key gluconeogenic enzymes. It also increases insulin sensitivity in muscle "
            "and fat tissue, enhances glucose uptake, and modestly reduces intestinal glucose "
            "absorption. Unlike sulfonylureas, metformin does not stimulate insulin secretion, "
            "giving it a low risk of hypoglycemia as monotherapy."
        ),
        "rejected": "Metformin lowers blood sugar. That is all you need to know about how it works.",
    },
]

preference_pairs = [
    create_preference_pair(item["instruction"], "", item["chosen"], item["rejected"])
    for item in PREFERENCE_TEMPLATES
]
pref_dataset = Dataset.from_list(preference_pairs)
pref_dataset.to_json(str(DATA_DIR / "preferences.jsonl"))
print(f"Created {len(pref_dataset)} preference pairs")
print(f"Saved to: {DATA_DIR / 'preferences.jsonl'}")

In [ ]:
# ============================================================
# Part 3b: DPO Training Config (skeleton)
# ============================================================

dpo_config = DPOConfig(
    output_dir=str(MODEL_DIR / "dpo-checkpoint"),
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    beta=0.1,                     # divergence from reference model
    max_prompt_length=512,
    max_length=1024,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    optim="adamw_8bit",
    logging_steps=10,
    report_to="wandb",
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    bf16=True,
    seed=SEED,
    remove_unused_columns=False,
)

print("[OK] DPO config ready.")
print(f"  beta: {dpo_config.beta}")
print(f"  learning rate: {dpo_config.learning_rate}")

# To train (uncomment):
# dpo_model = AutoModelForCausalLM.from_pretrained(
#     str(MODEL_DIR / "qlora-adapter"), torch_dtype=torch.bfloat16, device_map="auto"
# )
# dpo_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR / "qlora-adapter"))
# dpo_tokenizer.pad_token = dpo_tokenizer.eos_token
#
# dpo_trainer = DPOTrainer(
#     model=dpo_model, args=dpo_config,
#     train_dataset=pref_dataset, eval_dataset=pref_dataset,
#     tokenizer=dpo_tokenizer,
# )
# dpo_trainer.train()
# dpo_trainer.save_model()

print("[INFO] DPO training skeleton ready.")

---
## Part 4: AWQ Model Quantization

AWQ (Activation-Aware Weight Quantization) compresses a model to 4-bit integers while preserving accuracy by protecting salient weight channels identified through activation magnitudes.

**Memory savings**: ~4x reduction (e.g., 16 GB FP16 -> ~4 GB INT4).

In [ ]:
# ============================================================
# Part 4a: AWQ Quantization Config (skeleton)
# ============================================================

MODEL_TO_QUANTIZE = "<YOUR_MODEL_PATH>"  # e.g., models/dpo-checkpoint
AWQ_OUTPUT_DIR = str(MODEL_DIR / "awq-quantized")

quant_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM",
}

print("AWQ Quantization Config:")
print(json.dumps(quant_config, indent=2))

# To quantize (uncomment):
# from awq import AutoAWQForCausalLM
# model = AutoAWQForCausalLM.from_pretrained(
#     MODEL_TO_QUANTIZE, safetensors=True, device_map="auto"
# )
# tokenizer = AutoTokenizer.from_pretrained(MODEL_TO_QUANTIZE, trust_remote_code=True)
# model.quantize(tokenizer, quant_config=quant_config, calib_data="pileval")
# model.save_quantized(AWQ_OUTPUT_DIR, safetensors=True)
# tokenizer.save_pretrained(AWQ_OUTPUT_DIR)
# print(f"[OK] AWQ quantized model saved to {AWQ_OUTPUT_DIR}")

print("[INFO] AWQ quantization skeleton ready.")

In [ ]:
# ============================================================
# Part 4b: Model Size Comparison
# ============================================================

def get_directory_size(path: str) -> float:
    """Calculate total size of a directory in GB."""
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if os.path.exists(fp):
                total += os.path.getsize(fp)
    return total / (1024 ** 3)

# Placeholder sizes -- replace with real measurements
placeholder_sizes = {
    "Base Model (FP16)": 14.8,
    "QLoRA Adapter Only": 0.16,
    "Merged SFT+DPO (FP16)": 14.9,
    "AWQ 4-bit Quantized": 4.2,
}

fig, ax = plt.subplots(figsize=(10, 5))
names = list(placeholder_sizes.keys()); sizes = list(placeholder_sizes.values())
colors = ["#607D8B", "#4CAF50", "#2196F3", "#FF9800"]
bars = ax.barh(names, sizes, color=colors, edgecolor="white", linewidth=1.2)
for bar, size in zip(bars, sizes):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
            f"{size:.2f} GB", va="center", fontsize=11, fontweight="bold")
ax.set_xlabel("Size (GB)"); ax.set_title("Model Size Comparison Across Stages")
ax.set_xlim(0, max(sizes) * 1.3); ax.grid(axis="x", alpha=0.3)
compression = placeholder_sizes["Merged SFT+DPO (FP16)"] / placeholder_sizes["AWQ 4-bit Quantized"]
ax.annotate(f"~{compression:.1f}x compression",
            xy=(placeholder_sizes["AWQ 4-bit Quantized"], 3),
            xytext=(max(sizes) * 0.6, 3.3),
            arrowprops=dict(arrowstyle="->", color="black", lw=1.5),
            fontsize=11, fontweight="bold", color="#D32F2F")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "model_size_comparison.png", dpi=150)
plt.show()

---
## Part 5: vLLM Inference Deployment

vLLM provides high-throughput serving with PagedAttention, continuous batching, and native AWQ support. It exposes an OpenAI-compatible API.

In [ ]:
# ============================================================
# Part 5a: vLLM Serving Configuration
# ============================================================

vllm_config = {
    "model": "<YOUR_AWQ_MODEL_PATH>",
    "host": "0.0.0.0",
    "port": 8000,
    "served_model_name": "<YOUR_MODEL_NAME>",
    "max_model_len": 4096,
    "max_num_seqs": 32,
    "gpu_memory_utilization": 0.90,
    "quantization": "awq",
    "dtype": "auto",
    "trust_remote_code": True,
}

config_dir = PROJECT_ROOT / "configs"
config_dir.mkdir(parents=True, exist_ok=True)
with open(config_dir / "vllm_config.yaml", "w") as f:
    yaml.dump(vllm_config, f, default_flow_style=False)
print("[OK] vLLM config saved.")

print(f"\nLaunch command:")
print(f"python -m vllm.entrypoints.openai.api_server \\")
print(f"    --model {vllm_config['model']} \\")
print(f"    --host {vllm_config['host']} --port {vllm_config['port']} \\")
print(f"    --quantization {vllm_config['quantization']} \\")
print(f"    --max-model-len {vllm_config['max_model_len']} \\")
print(f"    --gpu-memory-utilization {vllm_config['gpu_memory_utilization']}")

In [ ]:
# ============================================================
# Part 5b: vLLM Offline Inference & OpenAI Client
# ============================================================

from vllm import LLM, SamplingParams

sampling_params = SamplingParams(
    temperature=0.7, top_p=0.95, top_k=50,
    max_tokens=512, repetition_penalty=1.1,
    stop=["<|im_end|>", "<|endoftext|>"],
)

test_prompts = [
    "<|im_start|>user\nWhat are the early warning signs of type 2 diabetes?<|im_end|>\n<|im_start|>assistant\n",
    "<|im_start|>user\nExplain how lisinopril works to lower blood pressure.<|im_end|>\n<|im_start|>assistant\n",
]

# To use vLLM offline (uncomment):
# llm = LLM(
#     model="<YOUR_AWQ_MODEL_PATH>", quantization="awq",
#     max_model_len=4096, gpu_memory_utilization=0.90,
# )
# outputs = llm.generate(test_prompts, sampling_params)
# for prompt, output in zip(test_prompts, outputs):
#     print(f"Prompt: {prompt[:80]}...")
#     print(f"Response: {output.outputs[0].text}\n")

print(f"Test prompts prepared: {len(test_prompts)}")
print("\nOpenAI-compatible client example (when vLLM server is running):")
print("""
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
response = client.chat.completions.create(
    model="<YOUR_MODEL_NAME>",
    messages=[{"role": "user", "content": "What is hypertension?"}],
    temperature=0.7, max_tokens=256,
)
print(response.choices[0].message.content)
""")

---
## Part 6: FastAPI Gateway with Auth, Rate Limiting & Routing

Production API gateway sitting in front of vLLM with API-key authentication, per-key rate limiting, health checks, and OpenAPI docs.

In [ ]:
# ============================================================
# Part 6: FastAPI Gateway (write to gateway/main.py)
# ============================================================

GATEWAY_CODE = '''
"""LLM API Gateway -- FastAPI with auth, rate limiting, and vLLM routing."""
import os, time, hashlib, logging
from fastapi import FastAPI, HTTPException, Request, Depends
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from fastapi.middleware.cors import CORSMiddleware
from slowapi import Limiter, _rate_limit_exceeded_handler
from slowapi.util import get_remote_address
from slowapi.errors import RateLimitExceeded
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import httpx

load_dotenv()

VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
API_KEYS = set(os.getenv("API_KEYS", "<YOUR_API_KEY_1>,<YOUR_API_KEY_2>").split(","))
REQUEST_TIMEOUT = int(os.getenv("REQUEST_TIMEOUT", "120"))

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

app = FastAPI(title="<YOUR_APP_NAME> LLM Gateway", version="1.0.0", docs_url="/docs")
limiter = Limiter(key_func=get_remote_address)
app.state.limiter = limiter
app.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True,
                   allow_methods=["*"], allow_headers=["*"])
security = HTTPBearer()

def verify_api_key(creds: HTTPAuthorizationCredentials = Depends(security)):
    if creds.credentials not in API_KEYS:
        raise HTTPException(status_code=403, detail="Invalid API key")
    return creds.credentials

class ChatMessage(BaseModel):
    role: str; content: str

class ChatCompletionRequest(BaseModel):
    messages: list[ChatMessage]
    temperature: float = 0.7
    max_tokens: int = 512
    top_p: float = 0.95

@app.get("/health")
async def health():
    try:
        async with httpx.AsyncClient(timeout=5.0) as c:
            backend_ok = (await c.get(f"{VLLM_BASE_URL}/models")).status_code == 200
    except Exception:
        backend_ok = False
    return {"status": "healthy" if backend_ok else "degraded", "backend": "connected" if backend_ok else "unreachable"}

@app.post("/v1/chat/completions")
@limiter.limit("<YOUR_RATE_LIMIT>")  # e.g., "100/minute"
async def chat_completions(request: Request, body: ChatCompletionRequest, api_key: str = Depends(verify_api_key)):
    rid = hashlib.md5(f"{time.time()}".encode()).hexdigest()[:12]
    logger.info(f"[{rid}] messages={len(body.messages)} max_tokens={body.max_tokens}")
    payload = {"model": "<YOUR_MODEL_NAME>", "messages": [m.model_dump() for m in body.messages],
               "temperature": body.temperature, "max_tokens": body.max_tokens, "top_p": body.top_p}
    try:
        async with httpx.AsyncClient(timeout=REQUEST_TIMEOUT) as c:
            resp = await c.post(f"{VLLM_BASE_URL}/chat/completions", json=payload)
            resp.raise_for_status()
            data = resp.json()
        return {"id": rid, "object": "chat.completion", "created": int(time.time()),
                "model": "<YOUR_MODEL_NAME>", "choices": data["choices"], "usage": data.get("usage", {})}
    except httpx.TimeoutException:
        raise HTTPException(status_code=504, detail="Backend timeout")
    except httpx.HTTPStatusError as e:
        raise HTTPException(status_code=502, detail=f"Backend error: {e.response.status_code}")

if __name__ == "__main__":
    import uvicorn; uvicorn.run(app, host="0.0.0.0", port=8080)
'''

gateway_dir = PROJECT_ROOT / "gateway"
gateway_dir.mkdir(parents=True, exist_ok=True)
with open(gateway_dir / "main.py", "w") as f:
    f.write(GATEWAY_CODE.strip())

# Gateway requirements
with open(gateway_dir / "requirements.txt", "w") as f:
    f.write("fastapi==0.115.0\nuvicorn==0.30.6\nslowapi==0.1.9\npython-dotenv==1.0.1\nhttpx==0.27.2\npydantic>=2.0.0\n")

# .env template
with open(gateway_dir / ".env.example", "w") as f:
    f.write("VLLM_BASE_URL=http://localhost:8000/v1\nAPI_KEYS=<YOUR_API_KEY_1>,<YOUR_API_KEY_2>\nREQUEST_TIMEOUT=120\n")

print("[OK] Gateway files created in gateway/")
print("  gateway/main.py")
print("  gateway/requirements.txt")
print("  gateway/.env.example")

---
## Part 7: Docker Containerization

Containerize the gateway and vLLM services with Docker Compose for reproducible deployment.

In [ ]:
# ============================================================
# Part 7: Dockerfile + docker-compose.yml
# ============================================================

docker_dir = PROJECT_ROOT / "docker"
docker_dir.mkdir(parents=True, exist_ok=True)

# --- Dockerfile.gateway ---
dockerfile_gateway = '''FROM python:3.11-slim
WORKDIR /app
RUN groupadd -r appuser && useradd -r -g appuser appuser
COPY gateway/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY gateway/ ./
USER appuser
EXPOSE 8080
HEALTHCHECK --interval=30s --timeout=10s --retries=3 \\
    CMD python -c "import httpx; httpx.get('http://localhost:8080/health').raise_for_status()"
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8080"]
'''

# --- Dockerfile.vllm ---
dockerfile_vllm = '''FROM vllm/vllm-openai:latest
ENV VLLM_MODEL_PATH=<YOUR_AWQ_MODEL_PATH>
ENV VLLM_MODEL_NAME=<YOUR_MODEL_NAME>
EXPOSE 8000
CMD ["--model", "${VLLM_MODEL_PATH}", "--host", "0.0.0.0", "--port", "8000",
     "--quantization", "awq", "--max-model-len", "4096",
     "--gpu-memory-utilization", "0.90", "--served-model-name", "${VLLM_MODEL_NAME}"]
'''

# --- docker-compose.yml ---
docker_compose = '''version: "3.8"
services:
  vllm:
    build:
      context: .
      dockerfile: docker/Dockerfile.vllm
    image: <YOUR_DOCKER_REGISTRY>/vllm:latest
    container_name: vllm-backend
    ports:
      - "8000:8000"
    volumes:
      - <YOUR_LOCAL_MODEL_DIR>:/models:ro
    environment:
      - VLLM_MODEL_PATH=/models/awq-quantized
      - VLLM_MODEL_NAME=<YOUR_MODEL_NAME>
      - CUDA_VISIBLE_DEVICES=0
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: 1
              capabilities: [gpu]
    restart: unless-stopped

  gateway:
    build:
      context: .
      dockerfile: docker/Dockerfile.gateway
    image: <YOUR_DOCKER_REGISTRY>/llm-gateway:latest
    container_name: api-gateway
    ports:
      - "8080:8080"
    environment:
      - VLLM_BASE_URL=http://vllm:8000/v1
      - API_KEYS=${API_KEYS}
      - REQUEST_TIMEOUT=120
    depends_on:
      vllm:
        condition: service_healthy
    restart: unless-stopped
'''

with open(docker_dir / "Dockerfile.gateway", "w") as f:
    f.write(dockerfile_gateway.strip())
with open(docker_dir / "Dockerfile.vllm", "w") as f:
    f.write(dockerfile_vllm.strip())
with open(PROJECT_ROOT / "docker-compose.yml", "w") as f:
    f.write(docker_compose.strip())

print("[OK] Docker files created:")
print("  docker/Dockerfile.gateway")
print("  docker/Dockerfile.vllm")
print("  docker-compose.yml")
print("\nBuild & run:")
print("  docker compose build")
print("  API_KEYS='<YOUR_KEY_1>,<YOUR_KEY_2>' docker compose up -d")

---
## Part 8: Auto-Evaluation: Base vs Fine-Tuned

Compare the base model against your fine-tuned model on a held-out test set using BLEU and ROUGE-L.

In [ ]:
# ============================================================
# Part 8a: Test Set Preparation
# ============================================================

TEST_QUESTIONS = [
    {"instruction": "What lifestyle changes can help manage type 2 diabetes?",
     "reference": ("Key lifestyle changes include balanced diet with controlled carbohydrate intake, "
                   "regular physical activity (150+ min/week), maintaining healthy weight, monitoring "
                   "blood glucose, quitting smoking, limiting alcohol, managing stress, and adequate sleep.")},
    {"instruction": "Describe how metformin works.",
     "reference": ("Metformin reduces hepatic glucose production by activating AMPK, increases insulin "
                   "sensitivity in peripheral tissues, enhances glucose uptake, and modestly reduces "
                   "intestinal glucose absorption. It carries a low risk of hypoglycemia.")},
    {"instruction": "What are risk factors for hypertension?",
     "reference": ("Risk factors include age, family history, obesity, physical inactivity, high sodium "
                   "diet, excessive alcohol, tobacco use, chronic stress, diabetes, and kidney disease.")},
    {"instruction": "Explain the difference between type 1 and type 2 diabetes.",
     "reference": ("Type 1 is autoimmune destruction of beta cells causing no insulin production; type 2 "
                   "is insulin resistance where cells don't respond effectively. Type 1 requires lifelong "
                   "insulin; type 2 can often be managed with oral medications and lifestyle.")},
    {"instruction": "What should a patient know about warfarin therapy?",
     "reference": ("Warfarin is an anticoagulant requiring regular INR monitoring. Dietary vitamin K "
                   "consistency is important. Many medications interact with it. Signs of bleeding require "
                   "immediate attention. Inform all healthcare providers before procedures.")},
]

test_dataset = Dataset.from_list(TEST_QUESTIONS)
test_dataset.to_json(str(DATA_DIR / "test_set.jsonl"))
print(f"Test set: {len(test_dataset)} questions")

In [ ]:
# ============================================================
# Part 8b: Evaluation Metrics
# ============================================================

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

def compute_bleu(reference: str, candidate: str) -> float:
    """Compute BLEU-4 with smoothing."""
    ref_tokens = [nltk.word_tokenize(reference.lower())]
    cand_tokens = nltk.word_tokenize(candidate.lower())
    return sentence_bleu(ref_tokens, cand_tokens, smoothing_function=SmoothingFunction().method4)

def compute_rouge_l(reference: str, candidate: str) -> float:
    """Compute ROUGE-L F1."""
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    return scorer.score(reference, candidate)["rougeL"].fmeasure

def evaluate_model(model_name: str, test_data: List[Dict], generate_fn) -> pd.DataFrame:
    """Evaluate a model on the test dataset."""
    results = []
    for i, item in enumerate(test_data):
        candidate = generate_fn(item["instruction"])
        reference = item["reference"]
        results.append({
            "example_id": i, "model": model_name,
            "instruction": item["instruction"][:60],
            "BLEU-4": round(compute_bleu(reference, candidate), 4),
            "ROUGE-L": round(compute_rouge_l(reference, candidate), 4),
        })
    return pd.DataFrame(results)

def mock_generate_fn(prompt: str) -> str:
    """Mock -- replace with real model inference."""
    return "This is a placeholder response for evaluation."

print("[OK] Evaluation functions defined.")
print("Replace mock_generate_fn with real model inference to run actual evaluation.")

In [ ]:
# ============================================================
# Part 8c: Run Evaluation & Visualize
# ============================================================

# Simulated results -- replace with real evaluation
sim_results = pd.DataFrame({
    "Model": ["Base Model"] * 5 + ["QLoRA Fine-Tuned"] * 5 + ["QLoRA + DPO"] * 5,
    "BLEU-4": [0.12, 0.15, 0.10, 0.14, 0.11,  0.22, 0.25, 0.19, 0.24, 0.21,  0.26, 0.28, 0.23, 0.27, 0.24],
    "ROUGE-L": [0.18, 0.21, 0.16, 0.20, 0.17, 0.28, 0.31, 0.25, 0.30, 0.27, 0.32, 0.34, 0.29, 0.33, 0.30],
})

agg = sim_results.groupby("Model").agg(
    BLEU_mean=("BLEU-4", "mean"), BLEU_std=("BLEU-4", "std"),
    ROUGE_mean=("ROUGE-L", "mean"), ROUGE_std=("ROUGE-L", "std"),
).reset_index()
print("Aggregate Results:")
print(agg.to_string(index=False))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=sim_results, x="Model", y="BLEU-4", palette="Blues", ax=axes[0])
sns.stripplot(data=sim_results, x="Model", y="BLEU-4", color="black", size=5, alpha=0.6, ax=axes[0])
axes[0].set_title("BLEU-4 Score by Model"); axes[0].tick_params(axis="x", rotation=15)
sns.boxplot(data=sim_results, x="Model", y="ROUGE-L", palette="Oranges", ax=axes[1])
sns.stripplot(data=sim_results, x="Model", y="ROUGE-L", color="black", size=5, alpha=0.6, ax=axes[1])
axes[1].set_title("ROUGE-L Score by Model"); axes[1].tick_params(axis="x", rotation=15)
plt.suptitle("Model Comparison: Base vs Fine-Tuned vs DPO-Aligned", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "evaluation_results.png", dpi=150, bbox_inches="tight")
plt.show()

base_bleu = agg[agg["Model"] == "Base Model"]["BLEU_mean"].values[0]
dpo_bleu  = agg[agg["Model"] == "QLoRA + DPO"]["BLEU_mean"].values[0]
print(f"\nBLEU improvement: {base_bleu:.3f} -> {dpo_bleu:.3f} (+{(dpo_bleu - base_bleu) / base_bleu * 100:.0f}%)")

---
## Part 9: Production Documentation Templates

Complete deployment, API reference, and troubleshooting docs for production handoff.

In [ ]:
# ============================================================
# Part 9: Generate Documentation
# ============================================================

DOCS_DIR = PROJECT_ROOT / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

deployment_md = """# Deployment Guide -- <YOUR_PROJECT_NAME>

## Prerequisites
- GPU Server: NVIDIA RTX 3090 / 4090 (24 GB VRAM)
- OS: Ubuntu 22.04 LTS; CUDA 12.1+; NVIDIA Driver 535+
- Docker 24+ with NVIDIA Container Toolkit; Python 3.10/3.11

## Quick Start
```bash
git clone <YOUR_REPO_URL> && cd <YOUR_PROJECT_DIR>
cp .env.example .env  # edit with your API_KEYS
docker compose build
API_KEYS='<YOUR_KEYS>' docker compose up -d
curl http://localhost:8080/health
```

## Manual Deployment
### 1. Start vLLM Server
```bash
python -m vllm.entrypoints.openai.api_server \\
    --model <YOUR_MODEL_PATH> --quantization awq \\
    --max-model-len 4096 --gpu-memory-utilization 0.90 --port 8000
```
### 2. Start API Gateway
```bash
cd gateway && pip install -r requirements.txt
uvicorn main:app --host 0.0.0.0 --port 8080
```

## Environment Variables
| Variable | Default | Description |
|----------|---------|-------------|
| VLLM_BASE_URL | http://localhost:8000/v1 | vLLM server URL |
| API_KEYS | (required) | Comma-separated valid API keys |
| REQUEST_TIMEOUT | 120 | Backend timeout (seconds) |
| RATE_LIMIT | 100/minute | Per-IP rate limit |
"""

api_md = """# API Reference -- <YOUR_PROJECT_NAME>

## Base URL: `http://<YOUR_SERVER_IP>:8080`

## Authentication
All endpoints except `/health` require: `Authorization: Bearer <YOUR_API_KEY>`

## Endpoints

### GET /health
Health check. Returns `{"status": "healthy", "backend": "connected"}`.

### POST /v1/chat/completions
OpenAI-compatible chat completions.

**Request:**
```json
{
  "messages": [{"role": "user", "content": "What is hypertension?"}],
  "temperature": 0.7,
  "max_tokens": 512
}
```

**Response:**
```json
{
  "id": "a1b2c3d4",
  "object": "chat.completion",
  "model": "<YOUR_MODEL_NAME>",
  "choices": [{"index": 0, "message": {"role": "assistant", "content": "..."}, "finish_reason": "stop"}],
  "usage": {"prompt_tokens": 45, "completion_tokens": 120, "total_tokens": 165}
}
```

### Error Codes
| Status | Meaning |
|--------|---------|
| 403 | Invalid API key |
| 429 | Rate limit exceeded |
| 502 | Backend error |
| 504 | Backend timeout |
"""

troubleshooting_md = """# Troubleshooting Guide -- <YOUR_PROJECT_NAME>

## CUDA Out of Memory
1. Reduce `gpu_memory_utilization` (try 0.80)
2. Reduce `max_model_len` or batch size
3. Enable CPU offloading: `--cpu-offload-gb 4`

## vLLM Fails to Start
1. Verify AWQ compatibility between autoawq and vLLM versions
2. Check quantization config matches model format
3. Try loading without quantization to isolate

## Gateway Returns 502
- Verify vLLM is running: `curl http://localhost:8000/health`
- Check VLLM_BASE_URL in .env
- In Docker Compose, use service name `vllm` (not localhost)

## Slow Inference
1. Enable prefix caching: `--enable-prefix-caching`
2. Increase `max_num_seqs`
3. Install Flash Attention 2

## NaN Loss During Training
1. Reduce learning rate
2. Add/increase gradient clipping
3. Check for corrupted data examples
"""

with open(DOCS_DIR / "DEPLOYMENT.md", "w") as f:
    f.write(deployment_md.strip())
with open(DOCS_DIR / "API.md", "w") as f:
    f.write(api_md.strip())
with open(DOCS_DIR / "TROUBLESHOOTING.md", "w") as f:
    f.write(troubleshooting_md.strip())

print("[OK] Documentation created:")
print("  docs/DEPLOYMENT.md")
print("  docs/API.md")
print("  docs/TROUBLESHOOTING.md")

---
## Final Project Delivery Checklist

### Dataset
- [ ] Domain-specific instruction dataset created (50+ examples)
- [ ] Data validation passed (no empty fields, unfilled placeholders)
- [ ] Train/validation split created (80/20)
- [ ] Preference pairs for DPO (5+ high-quality pairs)
- [ ] Test set held out for evaluation (5+ questions)

### Training
- [ ] QLoRA fine-tuning completed and converged
- [ ] Training logs saved (wandb or tensorboard)
- [ ] LoRA adapter weights saved to `models/qlora-adapter/`
- [ ] Training loss curves plotted and reviewed
- [ ] DPO alignment completed (optional but recommended)
- [ ] Final model checkpoint saved

### Quantization
- [ ] AWQ quantization completed successfully
- [ ] Quantized model loads without errors
- [ ] Model size comparison documented
- [ ] Quantization accuracy validated (spot-check outputs)

### Serving
- [ ] vLLM server starts and responds to requests
- [ ] Model loads with correct quantization settings
- [ ] OpenAI-compatible endpoint verified (`/v1/models`)
- [ ] Inference speed within SLA (< 10 s for 256 tokens)

### Gateway
- [ ] FastAPI gateway starts without errors
- [ ] API key authentication works (valid accepted, invalid rejected)
- [ ] Rate limiting active (429 response after limit exceeded)
- [ ] Health check endpoint returns correct status
- [ ] OpenAPI docs accessible at `/docs`

### Containerization
- [ ] Dockerfile.gateway builds successfully
- [ ] Dockerfile.vllm builds successfully
- [ ] `docker compose up` starts both services
- [ ] Health checks pass for both containers
- [ ] GPU accessible inside vLLM container

### Evaluation
- [ ] Evaluation script runs end-to-end
- [ ] BLEU and ROUGE scores computed for all models
- [ ] Base vs fine-tuned comparison visualized
- [ ] Improvement quantified (% gain over baseline)

### Documentation
- [ ] DEPLOYMENT.md complete with all sections filled
- [ ] API.md complete with endpoint documentation
- [ ] TROUBLESHOOTING.md covers common issues
- [ ] README.md has project overview and quick start
- [ ] All `<YOUR_...>` placeholders replaced with actual values

### Code Quality
- [ ] No hardcoded secrets (API keys in env vars)
- [ ] Requirements files up to date (`pip freeze` verified)
- [ ] Code is commented and follows consistent style
- [ ] `.gitignore` excludes model weights, venv, and secrets

### Production Readiness
- [ ] Error handling implemented throughout
- [ ] Logging configured with appropriate levels
- [ ] Health checks for all services
- [ ] Graceful shutdown handling
- [ ] Resource limits configured (Docker memory/CPU limits)
- [ ] Backup strategy for model weights documented

---

### Congratulations!

Once all boxes are checked, you have completed a full end-to-end LLM fine-tuning and deployment pipeline. This project demonstrates competency in all four stages:

1. **Foundations**: Transformer architecture, tokenization, attention
2. **Training & Fine-Tuning**: QLoRA, SFT, DPO alignment
3. **Optimization**: AWQ quantization, inference optimization, VRAM management
4. **Production**: API design, containerization, monitoring, documentation

**Next steps**: Deploy to a cloud GPU instance (AWS g5.xlarge, Lambda Labs, RunPod), set up CI/CD, add a web UI, or scale to multi-GPU serving.